# Caso E · 01 EDA ERA5 Xàtiva (mock)

> _Tutorial · Caso de uso: **E — Meteo & solar** · Capa Medallion: **bronce** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Conocer la estructura de ERA5 (mock 30 días Xàtiva) y derivar variables útiles para Caso B (forecast eléctrico) y Caso H (chatbot meteorológico).


## 2. Qué se aprende

- Variables ERA5 horarias: T_air, GHI, viento, lluvia, presión.
- Cómo descargar ERA5 real (CDS API) — fuera del repo.
- Conversiones unitarias: K→°C, J/m²→W/m², m→mm.
- Estacionalidad y diurnal cycle.


## 3. Contexto del caso de uso

ERA5 es el reanálisis climático global del ECMWF. Para Xàtiva tenemos un mock con 30 días horarios; el dataset real cubre desde 1940.


## 4. Relación con CENTINELA+

El dominio `weather_station/xativa/era5_gridpoint` complementa los edificios con variables externas correlacionadas.


## 5. Relación con Medallion

Bronce: NetCDF / mock CSV. Plata: `captia_point` con domain `weather_station`.


## 6. Datos de entrada

`era5_xativa_mock.csv`.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

Tags: `captia_env=dev`, `domain_id=weather_station`, `site_id=xativa`, `asset_id=era5_gridpoint`, `variable ∈ {temperature_outdoor, solar_irradiance, ...}`.


## 9. Carga de datos o mock

Cargamos mock.


In [2]:
csv_path = ROOT / "notebooks" / "_data" / "era5_xativa_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"])
df.head()


,timestamp,t_air_c,ghi_w_m2,wind_speed_ms,precip_mm,pressure_hpa
0,2024-06-01 00:00:00+00:00,19.82,0.0,1.00,0.00,1016.7
1,2024-06-01 01:00:00+00:00,18.68,0.0,2.54,0.00,1015.6
2,2024-06-01 02:00:00+00:00,21.06,0.0,0.00,0.01,1015.6
3,2024-06-01 03:00:00+00:00,22.21,0.0,4.16,0.00,1014.5
4,2024-06-01 04:00:00+00:00,20.56,0.0,1.78,0.01,1018.8


## 10. Exploración paso a paso

Estadísticas básicas y diurnal cycle.


In [3]:
print(df.describe().round(2))
df["hour"] = df["timestamp"].dt.hour
df.groupby("hour")[["t_air_c", "ghi_w_m2"]].mean().round(2)


       t_air_c  ghi_w_m2  wind_speed_ms  precip_mm  pressure_hpa
count   720.00    720.00         720.00     720.00        720.00
mean     25.83    228.88           1.92       0.02       1015.12
std       4.40    302.46           1.35       0.04          2.27
min      17.02      0.00           0.00       0.00       1009.00
25%      21.73      0.00           0.87       0.00       1013.70
50%      25.77      0.00           1.94       0.00       1015.10
75%      29.94    475.00           2.81       0.03       1016.60
max      33.78    950.00           6.37       0.28       1021.70


,t_air_c,ghi_w_m2
hour,,
0,19.58,0.00
1,20.07,0.00
2,20.86,0.00
3,21.58,0.00
4,22.80,0.00
5,23.98,0.00
6,25.62,0.00
7,27.32,178.17
8,28.92,342.00


## 11. Transformación bronce → plata

Notebook siguiente.


## 12. Construcción de capa oro

Notebook 03+04.


## 13. Visualizaciones explicativas

T y GHI horarios.


In [4]:
fig, ax1 = plt.subplots(figsize=(10, 3))
ax1.plot(df["timestamp"], df["t_air_c"], color="#FF5722", label="T")
ax1.set_ylabel("°C")
ax2 = ax1.twinx()
ax2.plot(df["timestamp"], df["ghi_w_m2"], color="#FFC107", label="GHI")
ax2.set_ylabel("W/m²")
plt.title("Diurnal — Xàtiva mock 30 días")
plt.tight_layout()


## 14. Validaciones

GHI nunca > 1100 W/m²; T entre -5 y 45.


In [5]:
assert df["ghi_w_m2"].between(0, 1100).all()
assert df["t_air_c"].between(-5, 50).all()
print("Rangos OK")


Rangos OK


## 15. Errores comunes

1. Confundir GHI con DNI (Direct Normal Irradiance).
2. Asumir que `pressure_hpa` es a nivel del mar (es local).
3. Promediar viento sin orientación (vector vs escalar).


## 16. Ejercicios propuestos

1. Calcula la insolación diaria (kWh/m²/día).
2. Compara `t_air_c` con la curva esperada para Csa.
3. Dibuja la rosa de los vientos.


## 17. Cómo se reutiliza con datos reales

Para descargar ERA5 real: registrarse en CDS, instalar `cdsapi`, ejecutar el script `scripts/era5_download.py` (no incluido).


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `05_case_E_weather_solar/02_bronze_to_silver_weather.ipynb`.
- Documento web del caso: `docs/use-cases/case-e-weather-solar.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo de irradiancia solar global (Iqbal 1983)

$$
G_h(t) = G_b(t) + G_d(t), \quad G_b(t) = G_{sc} \cdot \tau_b(t) \cdot \cos\theta_z(t)
$$

con $G_{sc} = 1361$ W/m² constante solar y $\theta_z$ ángulo cenital del sol:

$$
\cos\theta_z = \sin\delta \sin\phi + \cos\delta \cos\phi \cos\omega
$$

donde $\delta$ es declinación solar, $\phi$ latitud (Xátiva 38.99°N) y
$\omega$ ángulo horario.

### Clear-sky index

$$
k_c(t) = \frac{G_h(t)}{G_{clear}(t)} \in [0, 1]
$$

separa astronomía (determinista) de meteorología (estocástica).

### Predictor XGBoost para FV

$$
\hat{P}(t+h) = P_{nominal} \cdot \eta_{panel} \cdot \text{XGB}(k_c(t), T_{out}, t_{hora}, t_{año})
$$

### Métrica Skill Score

$$
\text{Skill} = 1 - \frac{\text{RMSE}_{model}}{\text{RMSE}_{persistence}}
$$

Objetivo Simarro: $\text{nMAE} \leq 8\%$ a 24 h, $\text{Skill} \geq 0.3$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Predicción solar permite optimizar despacho energético en centros con FV instalada y planificar climatización aprovechando picos de radiación. CAPTIA puede ofrecer este caso como **producto añadido** a centros con paneles.

### ROI estimado

| Concepto | Valor |
|---|---|
| Optimización despacho FV (centro 50 kWp) | +800 €/año |
| Sinergia con Caso B forecast | +500 €/año |
| Coste integración ERA5+AEMET | -1 200 € one-time |
| **Payback** | **~12 meses** |


## 21. Bibliografía y referencias

- Iqbal, M. (1983). *An Introduction to Solar Radiation*. Academic Press.
- ECMWF (2024). *ERA5 Reanalysis Documentation*. Copernicus Climate Change Service.
- AEMET. *Open Data Portal*. https://opendata.aemet.es
- Holmgren, W. F. et al. (2018). *pvlib python: a python package for modeling solar energy systems*. JOSS 3(29).
